<a href="https://colab.research.google.com/github/elena1346/Practica-01/blob/main/ExplicacionPractica1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción a HuggingFace

En este notebook vamos a ver algunas aplicaciones de procesamiento de lenguaje natural usando la librería [transformers](https://huggingface.co/docs/transformers/index) de [HuggingFace](https://huggingface.co).

Los ejemplos aquí presentandos están inspirados en el libro [Natural Language Processing with Transformers](https://www.oreilly.com/library/view/natural-language-processing/9781098136789/).

En primer lugar comenzamos instalando la librería.

In [3]:
!pip install "transformers[sentencepiece]<5" -Uq

La librería transformers proporciona una API en capas que permite interactuar con la librería a distintos niveles de abstracción. En este notebook usaremos las *pipelines*, que abstraen todos los pasos necesarios para convertir texto plano en un conjunto de predicciones a partir de un modelo.

In [5]:
from transformers import pipeline
import pandas as pd

## Aprende a usar pipeline

La documentación de "pipeline" la puedes encontrar en el siguiente link.
Presta especial atención a los parámetros formales de "pipeline", es decir, a los valores que puede tomar "task", "model", "config", etc.

https://huggingface.co/docs/transformers/main_classes/pipelines

En las siguientes celdas, puedes jugar a cambiar los parámetros para intentar entender lo que hace cada uno.

## Clasificación de sentimientos

Vamos a trabajar ahora usando pipeline en el siguiente texto.

In [6]:
text = """Estimado Amazon, la semana pasada pedí una camiseta del Barcelona en su tienda online de Alemania.
Desafortunadamente cuando abrí el paquete descubrí que había recibido una camiseta del Real Madrid. Como
seguidor del Barcelona, esto es un problema y solicito una devolución de mi dinero. Espero recibir respuesta pronto.
Un saludo, Jónathan
"""

El mensaje anterior puede ser el típico mensaje dejado en una página de valoraciones, y, por lo tanto, nos puede interesar saber si el feedback que estamos recibiendo es positivo o negativo. Esto se conoce como análisis de sentimientos, y se puede averiguar usando HuggingFace. Para ello vamos a usar un modelo del [Hub](https://huggingface.co/models).

In [7]:
classifier = pipeline(task='text-classification',model='pysentimiento/robertuito-sentiment-analysis')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

Device set to use cpu


Ahora podemos a partir de nuestro modelo generar una predicción. Las predicciones son siempre un diccionario de Python por lo que podemos mostrarlas de manera sencilla con Pandas.

In [9]:
outputs = classifier(text)
pd.DataFrame(outputs)

,label,score
0,NEG,0.738837


In [10]:
classifier(text)

[{'label': 'NEG', 'score': 0.7388370037078857}]

In [11]:
classifier('Me ha encantado este producto')

[{'label': 'POS', 'score': 0.9748007655143738}]

In [12]:
classifier('Me ha parecido horrible el producto')

[{'label': 'NEG', 'score': 0.9860497117042542}]

En la salida anterior, podemos ver que el sentimiento de la valoración es negativo.

## Reconocimiento de entidades

Predecir el sentimiento de un mensaje de un cliente, puede ser un buen primer paso, pero en este contexto es habitual tener que averiguar a qué servicio o producto se está refiriendo el cliente.

En procesado de lenguaje natural, los objetos reales como productos, lugares, o personas se conocen como entidades nombradas (en inglés *named entities*) y su extracción se puede realizar mediante una técnica llamada *named entity recognition*.

Desde el punto de vista de la librería transformers mediante los pipelines el uso de modelos para esta tarea es análogo a la clasificación de sentimientos.

In [13]:
ner_taggert = pipeline(model="mrm8488/bert-spanish-cased-finetuned-ner",aggregation_strategy="simple")
outputs = ner_taggert("Me compré una camiseta del Barcelona en Barcelona con Juan Maria")
pd.DataFrame(outputs)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,entity_group,score,word,start,end
0,ORG,0.994226,Barcelona,27,36
1,LOC,0.998673,Barcelona,40,49
2,PER,0.997415,Juan Maria,54,64


In [14]:
outputs = ner_taggert(text)
pd.DataFrame(outputs)

,entity_group,score,word,start,end
0,ORG,0.750501,Amazon,9,15
1,ORG,0.998589,Barcelona,56,65
2,LOC,0.999889,Alemania,89,97
3,ORG,0.999624,Real Madrid,186,197
4,ORG,0.997782,Barcelona,217,226
5,PER,0.997652,Jónathan,332,340


Podemos ver que se han detectado distintas entidades como organizaciones (Amazon, Barcelona, Real Madrid), personas (Jónathan) y también lugares (Alemania).

## Respondiendo preguntas

Ahora nos podemos preguntar si es posible responder preguntas sobre un texto. Esto es lo que se conoce como *question answering*. Para ello debemos proporcionar una pregunta y el contexto de dicha pregunta.

In [15]:
reader = pipeline(model="mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")
question = "¿Qué quiere el cliente?"
outputs = reader(question=question, context=text)
pd.DataFrame([outputs])

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of the model checkpoint at mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


,score,start,end,answer
0,0.161049,43,51,camiseta


Como podemos ver la respuesta que nos da, no es excesivamente buena. Vamos a ver cómo solucionar esto utilizando un modelo en inglés, para ello será necesario realizar una traducción de nuestro texto.

## Traducción

La traducción es una tarea muy habitual en nuestro día a día. Para ella podemos usar Google Translate o también podemos usar la librería de HuggingFace.

In [16]:
translator_es_en = pipeline(model="Helsinki-NLP/opus-mt-es-en")
outputs = translator_es_en(text,clean_up_tokenization_spaces=True,min_length=100)
translated_text = outputs[0]['translation_text']
print(translated_text)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


Dear Amazon, last week I ordered a Barcelona T-shirt from your online store in Germany. Unfortunately when I opened the package I discovered that I had received a Real Madrid T-shirt. As a follower of Barcelona, this is a problem and I request a refund of my money. I hope to receive an answer soon. Greetings, Jónathan. I hope you will receive a reply soon. I hope that you will receive your reply. I hope you will receive your reply.


Ahora podemos ir combinando pipelines para producir una respuesta más satisfactoria que la vista anteriormente.

In [17]:
outputs = translator_es_en(question,clean_up_tokenization_spaces=True)
translated_question = outputs[0]['translation_text']
print(translated_question)

What does the client want?


In [19]:
reader1 = pipeline("question-answering")
outputs1 = reader1(question=translated_question,context=translated_text)
pd.DataFrame([outputs1])

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


,score,start,end,answer
0,0.672192,244,264,a refund of my money


In [20]:
translator_en_es = pipeline(model="Helsinki-NLP/opus-mt-en-es")
outputs = translator_en_es(outputs1['answer'],clean_up_tokenization_spaces=True)
translated_answer = outputs[0]['translation_text']
print("\n", translated_answer)

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu



 un reembolso de mi dinero


## Resúmenes

El objetivo de la tarea de *summarization* es a partir de un texto, generar una versión corta del mismo que mantenga la información relevante.

In [24]:
summarizer = pipeline(model="josmunpen/mt5-small-spanish-summarization")
outputs = summarizer(text,max_length=45,clean_up_tokenization_spaces=True)
print(outputs[0]['summary_text'])

Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=45) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


amazon recibe una camiseta del madrid en su tienda online


Como podemos observar, el resumen no es especialmente bueno. Esto de nuevo lo podríamos resolver traduciendo al inglés.

## Generación de textos

La última aplicación que vamos a ver es la generación de textos. Esto lo podríamos utilizar para generar una respuesta automática al cliente.

In [23]:
generator = pipeline(model="DeepESP/gpt2-spanish")
respuesta = "Estimado Jónathan, siento oir que su pedido fue confundido"
prompt = "\n" + text + "\n\nRespuesta del comercial:\n"+ respuesta
outputs = generator(prompt,
                    max_new_tokens=60, #respuesta corta
                    truncation=True)
print(outputs[0]['generated_text'])

config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/261M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



Estimado Amazon, la semana pasada pedí una camiseta del Barcelona en su tienda online de Alemania.
Desafortunadamente cuando abrí el paquete descubrí que había recibido una camiseta del Real Madrid. Como
seguidor del Barcelona, esto es un problema y solicito una devolución de mi dinero. Espero recibir respuesta pronto.
Un saludo, Jónathan


Respuesta del comercial:
Estimado Jónathan, siento oir que su pedido fue confundido, pero lo que me interesa es

Jónathan: 

¿A qué se dedica? 

Lo hago como si fuera un agente del orden público. 

Un agradecimiento y una amistad

Un agradecimiento a la gente

A mi compañera,

A mi amiga,
